In [2]:
%pip install -q scikit-learn

from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# M5 leakage-safe preprocessing for predicting most_severe_injury.
# The cell produces both categorical data (CatBoost / FT-Transformer) and
# one-hot data (XGBoost / conventional sklearn models).

TARGET = "most_severe_injury"
TEST_SIZE = 0.20
RANDOM_STATE = 42
RARE_MIN_COUNT = 100

CATEGORICAL_FEATURES = [
    "traffic_control_device",
    "weather_condition",
    "lighting_condition",
    "first_crash_type",
    "trafficway_type",
    "alignment",
    "roadway_surface_cond",
    "road_defect",
    "intersection_related_i",
    "prim_contributory_cause",
]

BASE_NUMERIC_FEATURES = [
    "num_units",
    "crash_hour",
    "crash_day_of_week",
    "crash_month",
]

ENGINEERED_NUMERIC_FEATURES = [
    "num_units_was_missing",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos",
]

NUMERIC_FEATURES = BASE_NUMERIC_FEATURES + ENGINEERED_NUMERIC_FEATURES
MODEL_FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

# These columns are deliberately unavailable to the model. crash_type directly
# states injury/no-injury status; damage and injuries_* are post-outcome data.
EXCLUDED_LEAKAGE_COLUMNS = [
    "crash_type",
    "damage",
    "injuries_total",
    "injuries_fatal",
    "injuries_incapacitating",
    "injuries_non_incapacitating",
    "injuries_reported_not_evident",
    "injuries_no_indication",
]

# Original five levels are grouped into three operational severity levels.
THREE_CLASS_MAPPING = {
    "NO INDICATION OF INJURY": "NO_INJURY",
    "REPORTED, NOT EVIDENT": "MINOR_INJURY",
    "NONINCAPACITATING INJURY": "MINOR_INJURY",
    "INCAPACITATING INJURY": "SEVERE_INJURY",
    "FATAL": "SEVERE_INJURY",
}

# Explicit severity order for the three-class target.
TARGET_MAPPING = {
    "NO_INJURY": 0,
    "MINOR_INJURY": 1,
    "SEVERE_INJURY": 2,
}

def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "raw" / "traffic_accidents.csv").exists():
            return candidate
    raise FileNotFoundError("Cannot find data/raw/traffic_accidents.csv from the current directory.")

def normalize_text(series):
    result = series.astype("string").str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
    placeholders = ["", "NAN", "NULL", "NONE", "UNKNOWN", "UNKNOWN/NA", "NOT APPLICABLE"]
    return result.replace(placeholders, pd.NA)

def add_time_features(frame):
    frame = frame.copy()
    frame["is_weekend"] = frame["crash_day_of_week"].isin([1, 7]).astype("int8")
    frame["hour_sin"] = np.sin(2 * np.pi * frame["crash_hour"] / 24)
    frame["hour_cos"] = np.cos(2 * np.pi * frame["crash_hour"] / 24)
    frame["day_of_week_sin"] = np.sin(2 * np.pi * (frame["crash_day_of_week"] - 1) / 7)
    frame["day_of_week_cos"] = np.cos(2 * np.pi * (frame["crash_day_of_week"] - 1) / 7)
    frame["month_sin"] = np.sin(2 * np.pi * (frame["crash_month"] - 1) / 12)
    frame["month_cos"] = np.cos(2 * np.pi * (frame["crash_month"] - 1) / 12)
    return frame

ROOT = find_project_root()
RAW_PATH = ROOT / "data" / "raw" / "traffic_accidents.csv"
OUTPUT_DIR = ROOT / "data" / "processed" / "M5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for stale_name in ["valid_tabular.csv", "valid_onehot.csv"]:
    stale_path = OUTPUT_DIR / stale_name
    if stale_path.exists():
        stale_path.unlink()

raw = pd.read_csv(RAW_PATH)
raw.columns = raw.columns.str.strip().str.lower().str.replace(" ", "_", regex=False).str.replace("-", "_", regex=False)

required = ["crash_date", TARGET] + CATEGORICAL_FEATURES + BASE_NUMERIC_FEATURES
missing_required = sorted(set(required) - set(raw.columns))
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

rows_raw = len(raw)
raw = raw.drop_duplicates().copy()
rows_after_deduplication = len(raw)

work = raw[required].copy()
work["crash_date"] = pd.to_datetime(work["crash_date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
for column in CATEGORICAL_FEATURES + [TARGET]:
    work[column] = normalize_text(work[column])
for column in BASE_NUMERIC_FEATURES:
    work[column] = pd.to_numeric(work[column], errors="coerce")

work = work.dropna(subset=["crash_date", TARGET]).copy()
unknown_targets = sorted(set(work[TARGET].dropna()) - set(THREE_CLASS_MAPPING))
if unknown_targets:
    raise ValueError(f"Unexpected target labels: {unknown_targets}")

work["original_most_severe_injury"] = work[TARGET]
work[TARGET] = work[TARGET].map(THREE_CLASS_MAPPING)

# Stratified 80/20 split. The test set is not used to fit preprocessing rules.
train, test = train_test_split(
    work,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=work[TARGET],
)
train = train.copy()
test = test.copy()
splits = {"train": train, "test": test}

if any(part.empty for part in splits.values()):
    raise ValueError({name: len(part) for name, part in splits.items()})

# Fit categorical missing/rare rules on train only.
rare_categories = {}
for column in CATEGORICAL_FEATURES:
    train_values = train[column].fillna("MISSING")
    counts = train_values.value_counts(dropna=False)
    rare_categories[column] = sorted(counts[counts < RARE_MIN_COUNT].index.astype(str).tolist())
    known_categories = set(counts.index.astype(str))
    rare_set = set(rare_categories[column])
    for name, part in splits.items():
        values = part[column].fillna("MISSING").astype(str)
        # Categories rare in train and categories unseen after train share a stable fallback.
        part[column] = values.where(values.isin(known_categories) & ~values.isin(rare_set), "OTHER_RARE")

# Fit numeric imputation values on train only.
numeric_medians = {}
for column in BASE_NUMERIC_FEATURES:
    numeric_medians[column] = float(train[column].median())
    for part in splits.values():
        if column == "num_units":
            part["num_units_was_missing"] = part[column].isna().astype("int8")
        part[column] = part[column].fillna(numeric_medians[column])

splits = {name: add_time_features(part) for name, part in splits.items()}

# Encode target and save categorical/tabular files for CatBoost and FT-Transformer.
tabular_outputs = {}
for name, part in splits.items():
    output = part[MODEL_FEATURES].copy()
    output["original_most_severe_injury"] = part["original_most_severe_injury"].values
    output[TARGET] = part[TARGET].values
    output[f"{TARGET}_encoded"] = part[TARGET].map(TARGET_MAPPING).astype("int8").values
    path = OUTPUT_DIR / f"{name}_tabular.csv"
    output.to_csv(path, index=False)
    tabular_outputs[name] = output

# Fit one-hot columns on train only, then align the test set to the train schema.
train_onehot_features = pd.get_dummies(
    tabular_outputs["train"][MODEL_FEATURES],
    columns=CATEGORICAL_FEATURES,
    dtype="int8",
)
onehot_columns = train_onehot_features.columns.tolist()

for name, output in tabular_outputs.items():
    if name == "train":
        encoded_features = train_onehot_features
    else:
        encoded_features = pd.get_dummies(output[MODEL_FEATURES], columns=CATEGORICAL_FEATURES, dtype="int8")
        encoded_features = encoded_features.reindex(columns=onehot_columns, fill_value=0)
    encoded = encoded_features.copy()
    encoded[f"{TARGET}_encoded"] = output[f"{TARGET}_encoded"].to_numpy()
    encoded.to_csv(OUTPUT_DIR / f"{name}_onehot.csv", index=False)

train_target_counts = tabular_outputs["train"][TARGET].value_counts().reindex(TARGET_MAPPING).fillna(0)
class_weights = (len(tabular_outputs["train"]) / (len(TARGET_MAPPING) * train_target_counts)).to_dict()

metadata = {
    "target": TARGET,
    "target_mapping": TARGET_MAPPING,
    "three_class_mapping": THREE_CLASS_MAPPING,
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "model_features": MODEL_FEATURES,
    "excluded_leakage_columns": EXCLUDED_LEAKAGE_COLUMNS,
    "split_rule": {
        "method": "stratified random train/test split",
        "train_fraction": 0.80,
        "test_fraction": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "stratify_by": TARGET,
    },
    "rows": {name: len(output) for name, output in tabular_outputs.items()},
    "raw_rows": rows_raw,
    "exact_duplicates_removed": rows_raw - rows_after_deduplication,
    "rare_min_count": RARE_MIN_COUNT,
    "rare_categories_from_train": rare_categories,
    "numeric_medians_from_train": numeric_medians,
    "onehot_feature_columns": onehot_columns,
    "class_weights_from_train": {label: float(weight) for label, weight in class_weights.items()},
}

with (OUTPUT_DIR / "preprocessing_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Raw rows: {rows_raw:,}; exact duplicates removed: {rows_raw - rows_after_deduplication:,}")
for name, output in tabular_outputs.items():
    print(f"{name:>5}: {len(output):,} rows | {output[TARGET].value_counts().to_dict()}")
print(f"Model features: {len(MODEL_FEATURES)} ({len(CATEGORICAL_FEATURES)} categorical, {len(NUMERIC_FEATURES)} numeric)")
print(f"One-hot feature count: {len(onehot_columns)}")
print("Leakage columns excluded:", EXCLUDED_LEAKAGE_COLUMNS)



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Program Files\Python312\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Output directory: D:\Code\LZU-INFO442-Traffic-Accident-Severity-Prediction\data\processed\M5
Raw rows: 209,306; exact duplicates removed: 31
train: 167,420 rows | {'NO_INJURY': 123814, 'MINOR_INJURY': 38075, 'SEVERE_INJURY': 5531}
 test: 41,855 rows | {'NO_INJURY': 30953, 'MINOR_INJURY': 9519, 'SEVERE_INJURY': 1383}
Model features: 22 (10 categorical, 12 numeric)
One-hot feature count: 130
Leakage columns excluded: ['crash_type', 'damage', 'injuries_total', 'injuries_fatal', 'injuries_incapacitating', 'injuries_non_incapacitating', 'injuries_reported_not_evident', 'injuries_no_indication']
